Alle imports

In [1]:
import torch
import os
import torch
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch.utils.data
import pandas as pd
from sklearn.model_selection import train_test_split
import shutil
import torchvision
from torchvision.transforms import (
    Compose,
    Lambda,
    RandomCrop,
    RandomHorizontalFlip,
    CenterCrop
)
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import cv2
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import mediapipe
import mlflow.pytorch
import optuna

In [2]:
NUM_FRAMES = 30
ROOT_FOLDER = "data/"
OUTPUT_FOLDER = "mp_landmarks/"
MEDIAPIPE_MODEL_PATH = "C:\\Users\\nikam\\ML2\\v1\\models\\hand_landmarker.task"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [3]:
BaseOptions = mediapipe.tasks.BaseOptions
HandLandmarker = mediapipe.tasks.vision.HandLandmarker
HandLandmarkerOptions = mediapipe.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mediapipe.tasks.vision.RunningMode

Funktion um landmarks für ein video zu extrahieren und speichern

In [4]:
def extract_landmarks(video_path : str, landmarker):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Video kann nicht geoeffnet werden: {video_path}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total_frames-1, NUM_FRAMES, dtype=int)
    
    landmarks = np.zeros((NUM_FRAMES, 21, 3), dtype=np.float32)
    for out, frame_i in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_i)
        ret, frame = cap.read()
        if not ret:
            break
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mediapipe.Image(image_format=mediapipe.ImageFormat.SRGB, data= img_rgb)

        results = landmarker.detect(mp_image)
        if results.hand_landmarks:
            lms = results.hand_landmarks[0]
            coords = np.array([[l.x, l.y, l.z] for l in lms], dtype=np.float32)
            landmarks[out] = coords
    
    cap.release()

    return landmarks

In [5]:
def normalize_landmarks(seq):
    """
    seq: (T, 21, 3)
    Centers each frame at wrist (landmark 0), scales by wrist→middle-MCP distance (landmark 9).
    Returns (T, 21, 3) with values roughly in [-1, 1].
    """
    seq = seq.copy()
    wrist = seq[:, 0:1, :]                        # (T, 1, 3)
    seq -= wrist                                   # center at wrist
    scale = np.linalg.norm(seq[:, 9, :], axis=-1, keepdims=True)  # (T, 1)
    scale = np.maximum(scale, 1e-6)[:, :, None]   # (T, 1, 1) avoid div-by-zero
    seq /= scale
    return seq

Landmark extraction happens here

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MEDIAPIPE_MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1
)
with HandLandmarker.create_from_options(options) as landmarker:
    for split, csv_path in [("train", "train.csv"), ("val", "val.csv"), ("test", "test.csv")]:
        df = pd.read_csv(csv_path)
        rows = []
        for _, row in df.iterrows():
            landmarks = extract_landmarks(row['path'], landmarker)
            landmarks = normalize_landmarks(landmarks)
            npy_name = row["path"].replace("/","_").replace("\\","_") + ".npy"
            npy_path = os.path.join(OUTPUT_FOLDER, npy_name)
            np.save(npy_path, landmarks)
            rows.append({"path": row["path"], "label": row["label"], "npy_path": npy_path})
        pd.DataFrame(rows).to_csv(f"landmarks_{split}.csv", index=False)
        print(f"{split}: {len(rows)} videos processed")

Dataset Class

In [6]:
class HandLandmarkDataset(Dataset):
    def __init__(self, path):
        self.df = pd.read_csv(path)
        self.label_map = {"geste_0":0, "geste_1":1, "geste_2":2,
                 "class_1":3, "class_2":4, "Gesture01":5,
                 "Gesture02":6, "hand_turn":7, "ok_sign":8, "thumb_up":9}

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = np.load(row['npy_path']).astype(np.float32)
        x = torch.from_numpy(seq.reshape(seq.shape[0], -1))
        y= self.label_map[row['label']]

        return x, torch.tensor(y, dtype=torch.long)

In [7]:
class LandmarkLSTM(nn.Module):
    def __init__(self, input_dim=63, hidden=128, layers=2, num_classes=10):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, layers,
                            batch_first=True, dropout=0.3, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(hidden * 2, num_classes)
        )
    def forward(self, x):          # x: (B, T, 63)
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :])

In [8]:
def train(EPOCHS : int):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = LandmarkLSTM().to(device=device)
    
    train_loader = DataLoader(HandLandmarkDataset("landmarks_train.csv"), batch_size=16, shuffle=True)
    val_loader = DataLoader(HandLandmarkDataset("landmarks_val.csv"), batch_size=16, shuffle=True)

    loss_criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
    best =0
    train_accs, val_accs = [], []
    for i in range(1, EPOCHS+1):
        model.train()
        train_loss, train_correct, train_total =0,0,0

        for videos, labels in train_loader:
            videos, labels = videos.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(videos)
            loss = loss_criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * videos.size(0)
            train_correct += (outputs.argmax(1)== labels).sum().item()
            train_total += videos.size(0)
        
        model.eval()
        val_correct, val_total = 0,0
        with torch.no_grad():
            for videos, labels in val_loader:
                videos, labels = videos.to(device), labels.to(device)
                outputs = model(videos)
                val_correct += (outputs.argmax(1)== labels).sum().item()
                val_total += labels.size(0)
        
        train_acc = train_correct / train_total
        val_acc   = val_correct / val_total * 100
        train_accs.append(train_acc * 100)
        val_accs.append(val_acc)

        if val_acc > best:
            best = val_acc
            torch.save(model.state_dict(), "models/mp_model.pth")

        if i % 10 ==0:
             print(f"Epoch {i:3d} | train acc: {train_acc*100:.1f}%  "
              f"val acc: {val_acc:.1f}%  (best: {best:.1f}%)")
    
    return train_accs, val_accs

In [9]:
# train_accs, val_accs = train(50)

In [10]:
# train_accs_pct = [acc * 100 for acc in train_accs]  # convert to %

# fig, ax = plt.subplots()
# ax.plot(range(1, len(train_accs) + 1), train_accs, label="Train Accuracy (%)")
# ax.plot(range(1, len(val_accs) + 1), val_accs, label="Val Accuracy (%)")
# ax.set_xlabel("Epoch")
# ax.set_ylabel("Accuracy (%)")
# ax.set_title("Train vs Validation Accuracy")
# ax.legend()
# plt.show()

trying out a new model

In [11]:
from torchvision.models import resnet101, ResNet101_Weights

In [12]:
class HandGestenClassifier(nn.Module):
    """Per-frame MLP encoder -> LSTM -> classifier, for landmark sequences.

    Input:  x of shape (B, T, 63)  -- T frames, 21 landmarks * 3 coords.
    Output: logits of shape (B, num_classes).
    """
    def __init__(self, num_classes=10, input_dim=63, embed_dim=64,
                 hidden_size=128, num_layers=1, dropout=0.5):
        super().__init__()

        # Per-frame encoder: nn.Linear acts on the last dim, so this maps
        # each frame's 63 coords -> embed_dim, with no reshaping needed.
        self.frame_encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, embed_dim),
            nn.ReLU(),
        )

        # LSTM reads the sequence of per-frame embeddings.
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,                            # input: (B, T, features)
            dropout=dropout if num_layers > 1 else 0.0,  # only applies between stacked layers
        )

        # Classifier head
        self.dropout = nn.Dropout(p=dropout)
        self.linear  = nn.Linear(hidden_size, num_classes)

    def forward(self, x):                 # x: (B, T, 63)
        x = self.frame_encoder(x)         # (B, T, embed_dim)
        _, (h_n, _) = self.lstm(x)        # h_n: (num_layers, B, hidden_size)
        x = h_n[-1]                       # (B, hidden_size) -- last layer's final hidden state
        x = self.dropout(x)
        return self.linear(x)             # (B, num_classes)

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def new_train(lr=1e-3, hidden_size=128, num_layers=1, dropout=0.5,
              num_epochs=50, trial=None):
    """Train HandGestenClassifier with the given hyperparameters.

    Logs params/metrics to the currently active MLflow run (started by the
    Optuna objective). Returns the best validation accuracy reached.
    If an Optuna `trial` is passed, reports per-epoch val acc for pruning.
    """
    # Log the hyperparameters of this run
    mlflow.log_params({
        "lr": lr,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "dropout": dropout,
        "num_epochs": num_epochs,
        "optimizer": "Adam",
    })

    model = HandGestenClassifier(
        num_classes=10, hidden_size=hidden_size,
        num_layers=num_layers, dropout=dropout,
    ).to(device)
    optimizer = torch.optim.Adam(params=model.parameters(), lr=lr)
    loss_criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    train_loader = DataLoader(HandLandmarkDataset("landmarks_train.csv"), batch_size=16, shuffle=True)
    val_loader = DataLoader(HandLandmarkDataset("landmarks_val.csv"), batch_size=16, shuffle=True)
    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for videos, labels in train_loader:
            videos, labels = videos.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(videos)
            loss = loss_criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss    += loss.item() * videos.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total   += videos.size(0)

        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for videos, labels in val_loader:
                videos, labels = videos.to(device), labels.to(device)
                outputs = model(videos)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total   += labels.size(0)

        train_acc = train_correct / train_total * 100
        val_acc   = val_correct / val_total * 100

        # Per-epoch metrics to MLflow
        mlflow.log_metric("train_loss", train_loss / train_total, step=epoch)
        mlflow.log_metric("train_acc",  train_acc,                step=epoch)
        mlflow.log_metric("val_acc",    val_acc,                  step=epoch)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "flow_model.pth")

        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | train acc: {train_acc:.1f}%  "
                  f"val acc: {val_acc:.1f}%  (best: {best_val_acc:.1f}%)")

        # Optuna pruning: report intermediate value and stop unpromising trials early
        if trial is not None:
            trial.report(val_acc, epoch)
            if trial.should_prune():
                mlflow.log_metric("best_val_acc", best_val_acc)
                raise optuna.TrialPruned()

    mlflow.log_metric("best_val_acc", best_val_acc)
    return best_val_acc

In [16]:
mlflow.set_experiment("handgesten_hpo")

N_TRIALS   = 20    # number of hyperparameter combinations to try
NUM_EPOCHS = 50    # epochs per trial (pruning stops weak trials early)


def objective(trial):
    # Search space
    lr          = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    hidden_size = trial.suggest_categorical("hidden_size", [64, 128, 256])
    num_layers  = trial.suggest_int("num_layers", 1, 3)
    dropout     = trial.suggest_float("dropout", 0.1, 0.6)

    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        best_val_acc = new_train(
            lr=lr, hidden_size=hidden_size, num_layers=num_layers,
            dropout=dropout, num_epochs=NUM_EPOCHS, trial=trial,
        )
    return best_val_acc


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
)

with mlflow.start_run(run_name="optuna_search"):
    study.optimize(objective, n_trials=N_TRIALS)

    # Record the best configuration on the parent run
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_val_acc", study.best_value)

print("Best val acc:", study.best_value)
print("Best params :", study.best_params)

[I 2026-06-09 23:14:35,160] A new study created in memory with name: no-name-6607cb7c-437d-4073-b2b7-eeacfb35bea7


Epoch  10 | train acc: 86.0%  val acc: 92.1%  (best: 93.4%)
Epoch  20 | train acc: 96.5%  val acc: 84.2%  (best: 100.0%)
Epoch  30 | train acc: 98.2%  val acc: 85.5%  (best: 100.0%)
Epoch  40 | train acc: 100.0%  val acc: 98.7%  (best: 100.0%)


[I 2026-06-09 23:15:08,041] Trial 0 finished with value: 100.0 and parameters: {'lr': 0.0005611516415334506, 'hidden_size': 64, 'num_layers': 1, 'dropout': 0.17799726016810133}. Best is trial 0 with value: 100.0.


Epoch  50 | train acc: 100.0%  val acc: 98.7%  (best: 100.0%)
Epoch  10 | train acc: 20.6%  val acc: 23.7%  (best: 25.0%)
Epoch  20 | train acc: 53.1%  val acc: 52.6%  (best: 52.6%)
Epoch  30 | train acc: 80.7%  val acc: 82.9%  (best: 82.9%)
Epoch  40 | train acc: 91.7%  val acc: 92.1%  (best: 96.1%)


[I 2026-06-09 23:15:34,050] Trial 1 finished with value: 97.36842105263158 and parameters: {'lr': 0.00013066739238053285, 'hidden_size': 64, 'num_layers': 1, 'dropout': 0.5849549260809972}. Best is trial 0 with value: 100.0.


...

Epoch  10 | train acc: 52.2%  val acc: 53.9%  (best: 53.9%)
Epoch  10 | train acc: 92.5%  val acc: 93.4%  (best: 94.7%)
Epoch  20 | train acc: 100.0%  val acc: 97.4%  (best: 100.0%)
Epoch  30 | train acc: 97.8%  val acc: 94.7%  (best: 100.0%)
Epoch  40 | train acc: 98.7%  val acc: 100.0%  (best: 100.0%)


[I 2026-06-09 23:20:06,929] Trial 18 finished with value: 100.0 and parameters: {'lr': 0.0025172862976710966, 'hidden_size': 128, 'num_layers': 1, 'dropout': 0.42233684263404275}. Best is trial 0 with value: 100.0.


Epoch  50 | train acc: 91.2%  val acc: 100.0%  (best: 100.0%)


[I 2026-06-09 23:20:14,841] Trial 19 pruned. 


Epoch  10 | train acc: 85.1%  val acc: 86.8%  (best: 86.8%)
Best val acc: 100.0
Best params : {'lr': 0.0005611516415334506, 'hidden_size': 64, 'num_layers': 1, 'dropout': 0.17799726016810133}
